In [1]:
import pandas as pd
import numpy as np
import pickle
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


# ------------------- CONFIGURACIÓN ------------------- #

COLUMNAS_MODELO = [
    "superficie_construida", "banos", "distrito_encoded", "habitaciones",
    "planta_numerica", "exterior", "antiguedad", "terraza", "garaje", "calefaccion"
]


# ------------------- CARGA DE DATOS Y ARTEFACTOS ------------------- #

df = pd.read_csv("rental_properties_clustered.csv")
df = df.dropna()

# Encoding por media del precio
df["distrito_encoded"] = df["distrito"].map(df.groupby("distrito")["price_eur_pm"].mean())

# Artefactos globales
with open("imputer_rental.pkl", "rb") as f:
    imputer = pickle.load(f)
with open("scaler_global_rental.pkl", "rb") as f:
    scaler = pickle.load(f)


# ------------------- PREPROCESADO ------------------- #

X = df[COLUMNAS_MODELO]
y = df["price_eur_pm"]

X_imputado = imputer.transform(X)
X_scaled = scaler.transform(X_imputado)


# ------------------- DIVISIÓN Y ENTRENAMIENTO ------------------- #

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)

model = Sequential([
    Dense(64, activation='relu', input_shape=(X_train.shape[1],)),
    Dense(32, activation='relu'),
    Dense(16, activation='relu'),
    Dense(1)
])

model.compile(optimizer='adam', loss='mean_squared_error', metrics=['mae'])

early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

history = model.fit(
    X_train, y_train,
    validation_split=0.2,
    epochs=200,
    batch_size=32,
    callbacks=[early_stop],
    verbose=1
)


# ------------------- EVALUACIÓN ------------------- #

y_pred = model.predict(X_test).ravel()
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"✅ MAE en test: {mae:,.2f} EUR")
print(f"✅ R² en test: {r2:.4f}")


# ------------------- GUARDADO ------------------- #

model.save("nn_model_global_rental.keras")

with open("../model/artifacts/rental_model_rrnn.pkl", "wb") as f:
    pickle.dump(model, f)

metrics = {"mae": mae, "mse": mse, "r2": r2}
with open("nn_model_metrics_global_rental.pkl", "wb") as f:
    pickle.dump(metrics, f)


c:\Users\Marta\anaconda3\Lib\site-packages\keras\src\layers\core\dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/200
44/44 ━━━━━━━━━━━━━━━━━━━━ 3s 11ms/step - loss: 8734753.0000 - mae: 2453.0583 - val_loss: 9230089.0000 - val_mae: 2436.9866
Epoch 2/200
44/44 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 8507098.0000 - mae: 2395.9402 - val_loss: 9092274.0000 - val_mae: 2415.6882
Epoch 3/200
44/44 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 9399648.0000 - mae: 2459.7217 - val_loss: 8483813.0000 - val_mae: 2325.0010
Epoch 4/200
44/44 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 8263045.0000 - mae: 2319.0513 - val_loss: 6775226.0000 - val_mae: 2059.4893
Epoch 5/200
44/44 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 5597113.5000 - mae: 1942.4829 - val_loss: 4050299.2500 - val_mae: 1533.0817
Epoch 6/200
44/44 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 2893404.5000 - mae: 1305.6818 - val_loss: 2135805.0000 - val_mae: 1052.5574
Epoch 7/200
44/44 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 1482505.1250 - mae: 915.2275 - val_loss: 1601884.6250 - val_mae: 840.0079
Epoch 8/200
44/44 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - los

In [2]:
# ------------------- PREDICCIÓN DEL MODELO ------------------- #

import numpy as np
import pandas as pd
import pickle
import tensorflow as tf

# Columnas requeridas
COLUMNAS_MODELO = [
    "superficie_construida", "banos", "distrito_encoded", "habitaciones",
    "planta_numerica", "exterior", "antiguedad", "terraza", "garaje", "calefaccion"
]

def predecir_precio_vivienda_nn(nueva_vivienda: dict) -> float:
    """
    Predice el precio de una vivienda usando el modelo de red neuronal global.

    Args:
        nueva_vivienda (dict): Diccionario con las claves de COLUMNAS_MODELO

    Returns:
        float: Precio estimado en euros
    """
    # Convertir a DataFrame
    df_nueva = pd.DataFrame([nueva_vivienda])
    df_nueva = df_nueva.reindex(columns=COLUMNAS_MODELO)  # Asegura el orden correcto

    # Cargar artefactos
    with open("imputer_rental.pkl", "rb") as f:
        imputer = pickle.load(f)
    with open("scaler_global_rental.pkl", "rb") as f:
        scaler = pickle.load(f)
    
    # Cargar modelo
    model = tf.keras.models.load_model("nn_model_global_rental.keras")

    # Preprocesamiento
    X_imputado = imputer.transform(df_nueva)
    X_scaled = scaler.transform(X_imputado)

    # Predicción
    y_pred = model.predict(X_scaled).ravel()[0]

    print(f"🏠 Precio estimado (NN): {y_pred:,.2f} EUR")
    return y_pred


In [5]:
nueva_vivienda = {
    "superficie_construida": 120,
    "banos": 2,
    "distrito_encoded": 2953.936170212766,
    "habitaciones": 3,
    "planta_numerica": 2,
    "exterior": 1,
    "antiguedad": 15,
    "terraza": 1,
    "garaje": 1,
    "calefaccion": 1
}

precio_estimado = predecir_precio_vivienda_nn(nueva_vivienda)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 123ms/step
🏠 Precio estimado (NN): 6,283.37 EUR
